In [18]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
llm = init_chat_model("google_genai:gemini-3-flash-preview", temperature=0)

In [5]:
llm = init_chat_model("google_genai:gemini-3-flash-preview", temperature=0)
response = llm.invoke("Hola, como estas ?")
response.text

'¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?'

In [6]:
response = llm.invoke("Como esta el clima hoy en bogota ?")
response.text

'Hoy en **Bogotá**, el clima está presentándose de la siguiente manera (datos para el 22 de mayo):\n\n*   **Temperatura:** La máxima será de unos **19°C** y la mínima bajará hasta los **10°C**.\n*   **Estado del cielo:** Estará mayormente **nublado** durante gran parte del día.\n*   **Lluvias:** Hay una **alta probabilidad de lluvias y chubascos**, especialmente durante la tarde y las primeras horas de la noche.\n*   **Humedad:** Es alta (alrededor del 75%), lo que puede hacer que la sensación térmica sea un poco más fría.\n\n**Recomendación:** Como es habitual en Bogotá, lo mejor es salir con **paraguas** y vestirse con "capas" (una chaqueta que abrigue), ya que el clima puede cambiar rápidamente de un momento a otro.'

In [8]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan.

Tus productos son :
- Teclado
- Mouse
- Monitor
- Alexa 5
- Escritorio
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que ofreces")
]
response = llm.invoke(messages)
response.text

'¡Hola! Es un gusto saludarte. Actualmente cuento con los siguientes productos disponibles para ayudarte a equipar tu espacio:\n\n1. **Teclado**\n2. **Mouse**\n3. **Monitor**\n4. **Alexa 5**\n5. **Escritorio**\n\n¿Te gustaría conocer más detalles sobre alguno de ellos o necesitas ayuda para elegir el ideal para ti?'

In [11]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Get the products that the store offers")
def get_products():
    """Get the products that the store offers"""
    url = "https://api.escuelajs.co/api/v1/products"
    request = requests.get(url)
    products = request.json()
    # products = [
    #     {"name": "Teclado", "price" : 400},
    #     {"name": "Mouse", "price" : 250},
    #     {"name": "Monitor", "price" : 320},
    #     {"name": "Alexa 5", "price" : 500},
    #     {"name": "Escritorio", "price" : 600},
    # ]
    
    return " | ".join([f"{product['title']} -> {product['price']}" for product in products])

In [12]:
get_products.invoke({})

'Futuristic Holographic Soccer Cleats -> 39 | yyyy -> 39 | Chic Summer Denim Espadrille Sandals -> 33 | Vibrant Runners: Bold Orange & Blue Sneakers -> 27 | Vibrant Pink Classic Sneakers -> 84 | Futuristic Silver and Gold High-Top Sneaker -> 68 | Futuristic Chic High-Heel Boots -> 36 | Elegant Patent Leather Peep-Toe Pumps with Gold-Tone Heel -> 70 | Elegant Purple Leather Loafers -> 1000 | Classic Blue Suede Casual Shoes -> 39 | Sleek Futuristic Electric Bicycle -> 22 | Sleek All-Terrain Go-Kart -> 37 | Sleek Olive Green Hardshell Carry-On Luggage -> 43 | Chic Transparent Fashion Handbag -> 61 | Ручка -> 159 | kangkung -> 39 | nasi padang -> 23 | ghhdf393939 -> 20000 | cita -> 30000 | New Product -> 10 | Test Product -> 60 | g -> 196 | Test Product 2 -> 90 | Cincin -> 100 | Tes -> 100 | Baso Tahu -> 20 | tes666 -> 666 | tes000 -> 1000 | teslagibang -> 1 | tes1290. -> 1 | Produk Cecep -> 120 | Baju Baru -> 10000 | bottle -> 145 | water bottle -> 134 | chain watch -> 1345 | dsfd -> 6 | 

In [13]:

@tool("get_wether", description="Get the wether of a city")
def get_wether(city : str):
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1")
    data = response.json()
    latitude = data['results'][0]['latitude']
    longitude = data['results'][0]['longitude']
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data = response.json()
    response = f"The wether in {city} is {data['current_weather']['temperature']}C with {data['current_weather']['windspeed']}k/m of winds."
    return response
        

In [16]:
get_wether.invoke({"city" : "ibague"})

'The wether in ibague is 22.4C with 6.0k/m of winds.'

In [22]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y dar el clima de una cuidad.

Tus tools son :
- get_products : para obtener los prodictos que afrece la tienda
- get_wether : para obtener el clima de una cuidad
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que ofreces")
]
llm_with_tools = llm.bind_tools([get_products,get_wether])
response = llm_with_tools.invoke(messages)
# response.pretty_print()
response.tool_calls

[{'name': 'get_products',
  'args': {},
  'id': 'c8bc6533-9b9e-4df5-96bb-3c1f912d0898',
  'type': 'tool_call'}]

In [23]:
messages = [
    ("system", system_prompt),
    ("user", "Hola")
]
response = llm_with_tools.invoke(messages)
# response.pretty_print()
response.text

'¡Hola! Soy tu asistente de ventas. ¿En qué puedo ayudarte hoy?\n\nPuedo mostrarte los **productos** que ofrecemos en nuestra tienda o decirte cuál es el **clima** en cualquier ciudad que necesites. ¡Tú dirás!'

In [25]:
messages = [
    ("system", system_prompt),
    ("user", "Como esta el clima en la capital de Colombia ?")
]
response = llm_with_tools.invoke(messages)
response.pretty_print()
response.tool_calls

================================== Ai Message ==================================

[]
Tool Calls:
  get_wether (3d659815-cf5f-451e-97bd-7c8bde54a17a)
 Call ID: 3d659815-cf5f-451e-97bd-7c8bde54a17a
  Args:
    city: Bogotá


[{'name': 'get_wether',
  'args': {'city': 'Bogotá'},
  'id': '3d659815-cf5f-451e-97bd-7c8bde54a17a',
  'type': 'tool_call'}]